In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.silver.silver_events_daily AS
WITH source_data AS (
  SELECT
    md5(concat_ws('||', 
      coalesce(cast(employee_key as string), ''), 
      coalesce(employee_full_name, ''), 
      coalesce(event_type, ''), 
      coalesce(swap_cell, ''), 
      coalesce(start_datetime, ''), 
      coalesce(end_datetime, ''), 
      _source_file
    )) AS event_key,
    
    CAST(employee_key AS INT) AS employee_key,
    employee_full_name,
    event_type,
    
    CASE
      WHEN swap_cell IS NOT NULL AND trim(swap_cell) != '' THEN upper(trim(swap_cell))
      WHEN (swap_cell IS NULL OR trim(swap_cell) = '') AND remarks IS NOT NULL AND trim(remarks) != '' THEN upper(trim(remarks))
      ELSE 'NO ASSIGNMENT'
    END AS swap_cell,

    -- Konwersja ze stringa ISO
    CAST(CAST(start_datetime AS TIMESTAMP) AS DATE) AS start_date,
    date_format(CAST(start_datetime AS TIMESTAMP), 'HH:mm:ss') AS start_time_str,
    CAST(start_datetime AS TIMESTAMP) AS start_datetime,
    
    CAST(CAST(end_datetime AS TIMESTAMP) AS DATE) AS end_date,
    date_format(CAST(end_datetime AS TIMESTAMP), 'HH:mm:ss') AS end_time_str,
    CAST(end_datetime AS TIMESTAMP) AS end_datetime,
    
    CAST(fte AS DOUBLE) AS fte,
    remarks,
    source,
    _source_file,
    _ingested_at AS _bronze_ingested_at
  FROM data_warehouse_factory.bronze.bronze_events
  WHERE start_datetime IS NOT NULL AND end_datetime IS NOT NULL
),

-- 1. Rozbicie zakresu dat na poszczególne dni kalendarzowe
expanded_days AS (
  SELECT 
    s.*,
    explode(sequence(s.start_date, s.end_date, interval 1 day)) AS activity_date
  FROM source_data s
),

-- 2. Dedykowana logika wyliczania godzin startu i końca dla każdego dnia produkcyjnego
calculated_hours AS (
  SELECT 
    event_key,
    employee_key,
    employee_full_name,
    event_type,
    swap_cell,
    
    start_date AS start_date_general,
    end_date AS end_date_general,
    activity_date AS production_date,
    
    -- Wyznaczenie godziny początkowej dla danego dnia:
    -- Jeśli to pierwszy dzień -> weź dokładną godzinę podaną przez użytkownika
    -- Jeśli to kolejny dzień -> zacznij od domyślnego początku zmiany 06:00:00
    CASE 
      WHEN activity_date = start_date THEN start_time_str
      ELSE '06:00:00'
    END AS daily_start_time_str,

    -- Wyznaczenie godziny końcowej dla danego dnia:
    -- Jeśli to ostatni dzień -> weź dokładną godzinę podaną przez użytkownika
    -- Jeśli to dzień pośredni -> zakończ na domyślnym końcu zmiany 14:00:00
    CASE 
      WHEN activity_date = end_date THEN end_time_str
      ELSE '14:00:00'
    END AS daily_end_time_str,
    
    fte,
    remarks,
    source,
    _source_file,
    _bronze_ingested_at
  FROM expanded_days
),

-- 3. Zbudowanie czystych Timestampów i odfiltrowanie pustych/błędnych odcinków
building_timestamps AS (
  SELECT
    event_key,
    employee_key,
    employee_full_name,
    event_type,
    swap_cell,
    start_date_general,
    end_date_general,
    production_date,
    
    to_timestamp(concat(cast(production_date as string), ' ', daily_start_time_str)) AS daily_event_start,
    to_timestamp(concat(cast(production_date as string), ' ', daily_end_time_str)) AS daily_event_end,
    
    fte,
    remarks,
    source,
    _source_file,
    _bronze_ingested_at
  FROM calculated_hours
),

-- 4. Końcowe wyliczenie czasów w godzinach i minutach
final_summary AS (
  SELECT 
    md5(concat_ws('||', event_key, cast(production_date as string))) AS daily_event_key,
    event_key AS parent_event_key,
    
    employee_key,
    employee_full_name,
    event_type,
    swap_cell,
    
    start_date_general,
    end_date_general,
    production_date,
    
    daily_event_start,
    daily_event_end,
    
    ROUND(timestampdiff(MINUTE, daily_event_start, daily_event_end) / 60.0, 2) AS duration_hours,
    ROUND(timestampdiff(MINUTE, daily_event_start, daily_event_end), 2) AS duration_minutes,
    
    fte,
    remarks,
    source,
    _source_file,
    _bronze_ingested_at,
    current_timestamp() AS _silver_ingested_at
  FROM building_timestamps
  WHERE daily_event_start < daily_event_end
)

SELECT * FROM final_summary;